# 03. Classical Survival Models

Train and evaluate Cox PH and Random Survival Forest models.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import RAW_DATA_DIR, DEFAULT_CANCER_TYPE
from src.data.preprocess import preprocess_pipeline
from src.models.cox_model import train_cox_model, evaluate_cox_model, get_feature_importance
from src.models.rsf_model import train_rsf_model, evaluate_rsf_model
from src.evaluation.survival_plots import plot_kaplan_meier
from src.evaluation.metrics import stratify_by_risk

%matplotlib inline

## 1. Preprocess Data

In [ ]:
# Run preprocessing pipeline
train_df, test_df = preprocess_pipeline(
    clinical_path=RAW_DATA_DIR / f"{DEFAULT_CANCER_TYPE}_clinical.csv",
    expression_path=RAW_DATA_DIR / f"{DEFAULT_CANCER_TYPE}_expression.csv",
    save_intermediate=True
)

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

In [ ]:
# Prepare X and y
feature_cols = [c for c in train_df.columns if c not in ['patient_id', 'OS_time', 'OS_status']]

X_train = train_df[feature_cols]
y_train = train_df[['OS_time', 'OS_status']]
X_test = test_df[feature_cols]
y_test = test_df[['OS_time', 'OS_status']]

print(f"Features: {len(feature_cols)}")

## 2. Cox Proportional Hazards Model

In [ ]:
# Train Cox model
cox_model = train_cox_model(X_train, y_train)

# Evaluate
cox_metrics = evaluate_cox_model(cox_model, X_test, y_test)
print(f"Cox Model - C-index: {cox_metrics['c_index']:.4f}")

In [ ]:
# Feature importance
importance = get_feature_importance(cox_model, top_n=10)
print("\nTop 10 features:")
print(importance)

In [ ]:
# Risk stratification
from src.models.cox_model import predict_risk

test_risk = predict_risk(cox_model, X_test)
risk_groups = stratify_by_risk(test_risk, n_groups=2)

# Plot KM curves
fig = plot_kaplan_meier(
    event_times=y_test['OS_time'].values,
    event_observed=y_test['OS_status'].values,
    groups=risk_groups,
    group_labels=['Low Risk', 'High Risk'],
    title='Cox Model - Risk Stratification'
)
plt.show()

## 3. Random Survival Forest

In [ ]:
# Train RSF model (requires scikit-survival)
try:
    rsf_model = train_rsf_model(X_train, y_train)
    rsf_metrics = evaluate_rsf_model(rsf_model, X_test, y_test)
    print(f"RSF Model - C-index: {rsf_metrics['c_index']:.4f}")
except ImportError as e:
    print(f"Skipping RSF: {e}")
    print("Install scikit-survival to use Random Survival Forest")

## 4. Model Comparison

In [ ]:
# Compare models
results = [
    {'model': 'Cox PH', 'c_index': cox_metrics['c_index']}
]

try:
    results.append({'model': 'RSF', 'c_index': rsf_metrics['c_index']})
except:
    pass

results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df)

# Bar plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(results_df['model'], results_df['c_index'])
ax.set_ylabel('C-index')
ax.set_title('Model Performance Comparison')
ax.set_ylim([0.5, 1.0])
ax.axhline(y=0.5, color='r', linestyle='--', label='Random')
ax.legend()
plt.show()

## Summary

- Trained Cox PH model
- Trained Random Survival Forest (if available)
- Compared model performance
- Next: Deep learning with DeepSurv